In [1]:
pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.7 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import os
import cv2
import shutil

bbox_file = "/kaggle/input/datasets/fireworksbads/project2/project2/anno/list_bbox_inshop.txt"   # change this
partition_file = "/kaggle/input/datasets/fireworksbads/project2/project2/eval/list_eval_partition.txt"

bbox_df = pd.read_csv(
    bbox_file,
    sep=r"\s+",
    skiprows=1
)
bbox_df.columns = bbox_df.columns.str.strip()

bbox_df[["x_1", "y_1", "x_2", "y_2"]] = bbox_df[["x_1", "y_1", "x_2", "y_2"]].astype(int)

In [3]:
split_df = pd.read_csv(
    partition_file,
    sep=r"\s+",
    header=0,
    names=["image_name", "item_id", "evaluation_status"]
)
split_df.columns = split_df.columns.str.strip()

In [4]:
df = bbox_df.merge(split_df[["image_name", "evaluation_status"]], on="image_name", how="inner")

print(df.head())
print(df["evaluation_status"].value_counts())

                                          image_name  clothes_type  pose_type  \
0  img/WOMEN/Blouses_Shirts/id_00000001/02_1_fron...             1          1   
1  img/WOMEN/Blouses_Shirts/id_00000001/02_2_side...             1          2   
2  img/WOMEN/Blouses_Shirts/id_00000001/02_3_back...             1          3   
3  img/WOMEN/Blouses_Shirts/id_00000001/02_4_full...             1          4   
4       img/WOMEN/Dresses/id_00000002/02_1_front.jpg             3          1   

   x_1  y_1  x_2  y_2 evaluation_status  
0   50   49  208  235           gallery  
1  119   48  136  234             query  
2   50   42  213  240           gallery  
3   82   30  162  129             query  
4   65   45  233  252             train  
evaluation_status
train      25882
query      14218
gallery    12612
Name: count, dtype: int64


In [6]:
source_base = "/kaggle/input/datasets/fireworksbads/project2/project2/img/img"
output_base = "/kaggle/working/yolo_dataset"

for split in ["train", "val"]:
    os.makedirs(f"{output_base}/images/{split}", exist_ok=True)
    os.makedirs(f"{output_base}/labels/{split}", exist_ok=True)

In [7]:
# keep only train rows
train_df_full = df[df["evaluation_status"] == "train"].copy()

# ✅ NO filtering — keep all classes

# create class map
unique_classes = sorted(train_df_full["clothes_type"].unique())
class_map = {cls: i for i, cls in enumerate(unique_classes)}

print("Num classes:", len(class_map))
print(class_map)

Num classes: 3
{np.int64(1): 0, np.int64(2): 1, np.int64(3): 2}


In [35]:
print(len(train_df_full))

25882


In [8]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df_full,
    test_size=0.1,
    random_state=42,
    stratify=train_df_full["clothes_type"]
)

In [9]:
def convert_bbox_to_yolo(row, img_path, label_path, class_map, append=False):
    img = cv2.imread(img_path)
    if img is None:
        return False

    h, w = img.shape[:2]

    x1, y1, x2, y2 = row["x_1"], row["y_1"], row["x_2"], row["y_2"]

    x_center = ((x1 + x2) / 2) / w
    y_center = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h

    class_id = class_map[row["clothes_type"]]

    mode = "a" if append else "w"

    with open(label_path, mode) as f:
        f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}\n")

    return True

In [10]:
def process_split(dataframe, split_name):
    global success, missing

    grouped = dataframe.groupby("image_name")

    for img_name, group in grouped:

        img_path = os.path.join(source_base, img_name)

        dst_img = os.path.join(output_base, "images", split_name, img_name)
        dst_label = os.path.join(output_base, "labels", split_name, img_name.replace(".jpg", ".txt"))

        os.makedirs(os.path.dirname(dst_img), exist_ok=True)
        os.makedirs(os.path.dirname(dst_label), exist_ok=True)

        if not os.path.exists(img_path):
            missing += 1
            continue

        shutil.copy(img_path, dst_img)

        first = True
        for _, row in group.iterrows():
            convert_bbox_to_yolo(row, img_path, dst_label, class_map, append=not first)
            first = False

In [ ]:
#process_split(train_df, "train")
#process_split(val_df, "val")

#print(f"Done! Prepared: {success} | Missing: {missing}")

In [74]:
row = train_df.iloc[0]
img_name = row["image_name"]
split_name = "train"

dst_img = os.path.join(output_base, "images", split_name, img_name)
print(dst_img)

/kaggle/working/yolo_dataset/images/train/img/WOMEN/Blouses_Shirts/id_00001165/01_7_additional.jpg


In [75]:
import glob

print("Train images:",
      len(glob.glob("/kaggle/working/yolo_dataset/images/train/**/*.jpg", recursive=True)))

print("Train labels:",
      len(glob.glob("/kaggle/working/yolo_dataset/labels/train/**/*.txt", recursive=True)))

Train images: 23293
Train labels: 23293


In [12]:
print("Train images:", train_df["image_name"].nunique())
print("Val images:", val_df["image_name"].nunique())

Train images: 23293
Val images: 2589


In [15]:
import os

label_dir = "/kaggle/working/yolo_dataset/labels/train"

classes = set()

for file in os.listdir(label_dir):
    if file.endswith(".txt"):
        with open(os.path.join(label_dir, file)) as f:
            for line in f:
                classes.add(line.split()[0])

print(classes)

set()


In [16]:
print("Total rows (objects):", len(df))
print("Total images:", df["image_name"].nunique())

Total rows (objects): 52712
Total images: 52712


In [17]:
print(df["clothes_type"].value_counts())

clothes_type
1    33487
2    10494
3     8731
Name: count, dtype: int64


In [18]:
class_map = {1: 0, 2: 1, 3: 2}

In [19]:
print(df[["clothes_type", "image_name"]].head(20))

    clothes_type                                         image_name
0              1  img/WOMEN/Blouses_Shirts/id_00000001/02_1_fron...
1              1  img/WOMEN/Blouses_Shirts/id_00000001/02_2_side...
2              1  img/WOMEN/Blouses_Shirts/id_00000001/02_3_back...
3              1  img/WOMEN/Blouses_Shirts/id_00000001/02_4_full...
4              3       img/WOMEN/Dresses/id_00000002/02_1_front.jpg
5              3        img/WOMEN/Dresses/id_00000002/02_2_side.jpg
6              3        img/WOMEN/Dresses/id_00000002/02_4_full.jpg
7              3  img/WOMEN/Dresses/id_00000002/02_7_additional.jpg
8              2        img/WOMEN/Skirts/id_00000003/02_1_front.jpg
9              2         img/WOMEN/Skirts/id_00000003/02_2_side.jpg
10             2         img/WOMEN/Skirts/id_00000003/02_3_back.jpg
11             2         img/WOMEN/Skirts/id_00000003/02_4_full.jpg
12             2   img/WOMEN/Skirts/id_00000003/02_7_additional.jpg
13             1  img/WOMEN/Blouses_Shirts/id_00

In [21]:
data_yaml = f"""path: {output_base}
train: images/train
val: images/val

names:
  0: clothes_type_1
  1: clothes_type_2
  2: clothes_type_3
"""
with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml)

In [22]:
print(open("/kaggle/working/data.yaml").read())


path: /kaggle/working/yolo_dataset
train: images/train
val: images/val

names:
  0: clothes_type_1
  1: clothes_type_2
  2: clothes_type_3



In [89]:
#from ultralytics import YOLO
#
#model = YOLO("yolov8n.pt")
#model.train(data="/kaggle/working/data.yaml", epochs=30, imgsz=640)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bc6bf1e5070>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [93]:
metrics = model.val(data="/kaggle/working/data.yaml")
print(metrics)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 556.4±101.7 MB/s, size: 17.7 KB)
val: Scanning /kaggle/working/yolo_dataset/labels/val/img/MEN/Denim/id_00000089.cache... 2589 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2589/2589 775.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 162/162 8.3it/s 19.6s0.1s
                   all       2589       2589      0.869      0.905      0.939      0.763
        clothes_type_1       1671       1671      0.906      0.935      0.958      0.771
        clothes_type_2        490        490      0.848      0.868      0.916      0.737
        clothes_type_3        428        428      0.852      0.912      0.944      0.779
Speed: 0.9ms preprocess, 3.6ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
ultralytics.utils.metrics.DetMetrics obje

In [23]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch
from PIL import Image

In [26]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, CLIPModel, CLIPProcessor

device0 = "cuda:0"
device1 = "cuda:1"

# BLIP-2 on GPU 0
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model_blip = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16
).to(device0)

# CLIP on GPU 1
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device1)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [37]:
from ultralytics import YOLO
from PIL import Image
import cv2

model_yolo = YOLO("/kaggle/input/datasets/fireworksbads/output/detect/train/weights/best.pt")

img_path = "/kaggle/input/datasets/fireworksbads/dataset/images/val/img/WOMEN/Dresses/id_00000059/02_1_front.jpg"

results = model_yolo(img_path)

img = cv2.imread(img_path)

for r in results:
    boxes = r.boxes.xyxy.cpu().numpy()

    for box in boxes:
        x1, y1, x2, y2 = map(int, box)

        crop = img[y1:y2, x1:x2]

        # convert to PIL
        crop_pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

        # now use BLIP
        inputs = processor(images=crop_pil, return_tensors="pt").to(device)
        generated_ids = model_blip.generate(**inputs, max_new_tokens=30)

        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        print(caption)


image 1/1 /kaggle/input/datasets/fireworksbads/dataset/images/val/img/WOMEN/Dresses/id_00000059/02_1_front.jpg: 640x640 1 clothes_type_1, 7.3ms
Speed: 13.4ms preprocess, 7.3ms inference, 32.6ms postprocess per image at shape (1, 3, 640, 640)
the black and pink floral print dress has a v - neckline and a long, open back



In [44]:
device_blip = "cuda:0"   # keep BLIP on GPU 0

# preprocessing
inputs = processor(
    images=crop_pil,
    return_tensors="pt"
)

# move to GPU + half precision
inputs = {k: v.to(device_blip, torch.float16) for k, v in inputs.items()}

# inference
with torch.no_grad():
    ids = model_blip.generate(
        **inputs,
        max_new_tokens=20
    )

# decode
caption = processor.batch_decode(ids, skip_special_tokens=True)[0]

print(caption)

the black and pink floral print dress has a v - neckline and a long, open back



In [49]:
clip_inputs = clip_processor(
    text=[caption],
    images=crop_pil,
    return_tensors="pt",
    padding=True
)

# move to GPU correctly (DON’T convert everything to float16)
clip_inputs = {
    "input_ids": clip_inputs["input_ids"].to(device1),                # keep int
    "attention_mask": clip_inputs["attention_mask"].to(device1),      # keep int
    "pixel_values": clip_inputs["pixel_values"].to(device1, torch.float16)  # ONLY this is fp16
}

with torch.no_grad():
    outputs = clip_model(**clip_inputs)

In [50]:

image_emb = outputs.image_embeds
text_emb = outputs.text_embeds

In [51]:
image_emb = image_emb / image_emb.norm(dim=-1, keepdim=True)
text_emb  = text_emb  / text_emb.norm(dim=-1, keepdim=True)

In [52]:
alpha = 0.7   # tune this later

fused_emb = alpha * image_emb + (1 - alpha) * text_emb

# normalize again
fused_emb = fused_emb / fused_emb.norm(dim=-1, keepdim=True)

In [54]:
embedding_db = []
embedding_db.append(fused_emb.cpu())

In [56]:
db_emb = torch.cat([item["embedding"] for item in embedding_db], dim=0)

/tmp/ipykernel_55/2293112206.py:1: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  db_emb = torch.cat([item["embedding"] for item in embedding_db], dim=0)


IndexError: too many indices for tensor of dimension 2

In [55]:
similarity = torch.matmul(query_emb, db_emb.T)

NameError: name 'query_emb' is not defined

In [138]:
import torch.nn.functional as F

final_emb = F.normalize(final_emb, dim=-1)

In [141]:
embeddings = []
embeddings.append(final_emb.cpu())

In [142]:
import glob

gallery_paths = glob.glob("/kaggle/input/datasets/fireworksbads/mapped/gallery/**/*.jpg", recursive=True)
query_paths   = glob.glob("/kaggle/input/datasets/fireworksbads/mapped/query/**/*.jpg", recursive=True)

In [143]:
print("Gallery images:", len(gallery_paths))
print("Query images:", len(query_paths))

Gallery images: 12612
Query images: 14218


In [145]:
from ultralytics import YOLO
from PIL import Image
import cv2
import torch
import torch.nn.functional as F

yolo = YOLO("runs/detect/train/weights/best.pt")

def process_image(img_path):
    img = cv2.imread(img_path)

    if img is None:
        return []

    # YOLO inference (faster)
    results = yolo(img_path, verbose=False)

    embeddings = []

    with torch.no_grad():   # 🔥 important

        for r in results:
            if r.boxes is None:
                continue

            boxes = r.boxes.xyxy.cpu().numpy()

            for box in boxes:
                x1, y1, x2, y2 = map(int, box)

                crop = img[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                crop_pil = Image.fromarray(
                    cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                )

                # 🔹 BLIP (keep on CPU)
                inputs = blip_processor(
                    images=crop_pil,
                    return_tensors="pt"
                )

                ids = model_blip.generate(**inputs, max_new_tokens=20)

                caption = blip_processor.batch_decode(
                    ids, skip_special_tokens=True
                )[0]

                # 🔹 CLIP (GPU)
                clip_inputs = clip_processor(
                    text=[caption],
                    images=crop_pil,
                    return_tensors="pt",
                    padding=True
                ).to(device)

                outputs = clip_model(**clip_inputs)

                img_emb = outputs.image_embeds
                txt_emb = outputs.text_embeds

                # 🔹 combine
                final_emb = 0.5 * img_emb + 0.5 * txt_emb

                # 🔥 normalize (VERY IMPORTANT)
                final_emb = F.normalize(final_emb, dim=-1)

                embeddings.append(final_emb.cpu())

    return embeddings

In [ ]:
from tqdm import tqdm

gallery_embeddings = {}

for path in tqdm(gallery_paths):
    gallery_embeddings[path] = process_image(path)

  9%|▉         | 1109/12612 [40:31<6:19:06,  1.98s/it] 